# IBKR Flex sync

Pulls the "Trade History API" Flex Query (Cash Report + Open Positions + Trades) and brings `data/brokers/ibkr/ledger.csv` up to date. Safe to re-run: ledger events dedupe by `event_id`, so running this twice in a row just confirms nothing changed.

Run this regularly — the underlying Flex Query is scoped to "Last Business Day" on IBKR's side, so a missed run creates a permanent gap rather than something you can ask for later. If a run raises `TradeHistoryGapError`, see `docs/ibkr_flex_api.md` for how to backfill it. All the mechanics (protocol, error codes, the ledger vs. the raw archive) are documented there too.

In [ ]:
from trades.brokers.ibkr import api, main, preprocessing
from trades.config import IbkrFlexApiConfig, IbkrFlexCredentials

credentials = IbkrFlexCredentials()  # reads IBKR_FLEX_WEB_SERVICE_TOKEN / IBKR_QUERY_ID from .env
config = IbkrFlexApiConfig()

## Run the sync

One network round trip (SendRequest, then poll GetStatement until ready), then the ledger is updated from that single fetched statement's `<Trade>` rows.

In [ ]:
result = main.sync_ibkr_account(credentials, config)
result

## What's in the cache now

In [ ]:
ledger = main.load_ledger(config)
first_date = ledger["event_datetime"].min().date()
last_date = ledger["event_datetime"].max().date()
print(f"{len(ledger)} ledger events cached, spanning {first_date} to {last_date}")
ledger